<a href="https://colab.research.google.com/github/sauravv-kunwar/100-days-of-ML/blob/main/Regression_Tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
from pandas_datareader import data
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import r2_score
from sklearn.datasets import fetch_california_housing # Changed from load_boston
from sklearn.model_selection import GridSearchCV

In [5]:
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['target'] = housing.target # Add the target variable to the DataFrame
display(df.head())

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


### 1. Prepare Data for Regression Tree
First, we need to separate the features (X) from the target variable (y) and then split the data into training and testing sets.

In [6]:
# Define features (X) and target (y)
X = df.drop('target', axis=1) # All columns except 'target' are features
y = df['target'] # 'target' column is the target variable

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (16512, 8)
X_test shape: (4128, 8)
y_train shape: (16512,)
y_test shape: (4128,)


### 2. Train a Decision Tree Regressor
Now, let's initialize and train a `DecisionTreeRegressor` model on the training data.

In [7]:
# Initialize the DecisionTreeRegressor
# We can specify parameters like max_depth to control tree complexity
dtr = DecisionTreeRegressor(random_state=42)

# Train the model
dtr.fit(X_train, y_train)

print("Decision Tree Regressor trained successfully!")

Decision Tree Regressor trained successfully!


### 3. Make Predictions and Evaluate the Model
After training, we'll use the model to make predictions on the test set and evaluate its performance using metrics like Mean Absolute Error (MAE), Mean Squared Error (MSE), and R-squared.

In [8]:
# Make predictions on the test set
y_pred = dtr.predict(X_test)

# Evaluate the model
mae = metrics.mean_absolute_error(y_test, y_pred)
mse = metrics.mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R2): {r2:.4f}")

Mean Absolute Error (MAE): 0.4547
Mean Squared Error (MSE): 0.4952
Root Mean Squared Error (RMSE): 0.7037
R-squared (R2): 0.6221


This example provides a basic implementation of a regression tree. You can further tune the model using techniques like GridSearchCV to find optimal hyperparameters (e.g., `max_depth`, `min_samples_leaf`).

### 4. Hyperparameter Tuning with GridSearchCV
We can use `GridSearchCV` to systematically work through multiple combinations of hyperparameters, cross-validating each combination to determine which set of parameters works best.

In [9]:
# Define the parameter grid to search
param_grid = {
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize GridSearchCV
# We'll use the DecisionTreeRegressor again, but this time it will be tuned.
grid_search = GridSearchCV(estimator=DecisionTreeRegressor(random_state=42),
                           param_grid=param_grid,
                           cv=5, # 5-fold cross-validation
                           scoring='neg_mean_squared_error',
                           n_jobs=-1, # Use all available cores
                           verbose=1)

# Fit GridSearchCV to the training data
grid_search.fit(X_train, y_train)

print("GridSearchCV training complete!")

Fitting 5 folds for each of 45 candidates, totalling 225 fits
GridSearchCV training complete!


### 5. Best Parameters and Model Evaluation
After the grid search is complete, we can inspect the best parameters found and evaluate the performance of the model with these optimal settings.

In [10]:
# Print the best parameters and best score
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Score (Negative MSE): {grid_search.best_score_:.4f}")

# The best score is negative MSE, so we convert it to positive RMSE for better interpretability
best_rmse = np.sqrt(-grid_search.best_score_)
print(f"Best RMSE: {best_rmse:.4f}")

# Evaluate the best model on the test set
best_dtr = grid_search.best_estimator_
y_pred_tuned = best_dtr.predict(X_test)

mae_tuned = metrics.mean_absolute_error(y_test, y_pred_tuned)
mse_tuned = metrics.mean_squared_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mse_tuned)
r2_tuned = r2_score(y_test, y_pred_tuned)

print(f"\nMetrics for Tuned Model on Test Set:")
print(f"Mean Absolute Error (MAE): {mae_tuned:.4f}")
print(f"Mean Squared Error (MSE): {mse_tuned:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_tuned:.4f}")
print(f"R-squared (R2): {r2_tuned:.4f}")

Best Parameters: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2}
Best Score (Negative MSE): -0.3873
Best RMSE: 0.6224

Metrics for Tuned Model on Test Set:
Mean Absolute Error (MAE): 0.4311
Mean Squared Error (MSE): 0.4084
Root Mean Squared Error (RMSE): 0.6391
R-squared (R2): 0.6883
